In [5]:
import yaml
from pathlib import Path
import copy
import sys

# Now we can import our trusted data source list
from common.config_wells import get_data_sources

# ==============================================================================
# 1. CAMPAIGN BLUEPRINT (Your Unchanged Template)
# ==============================================================================
# This template defines the structure and default values for all campaigns.
# Your working `infra` paths are preserved exactly as you requested.
CAMPAIGN_TEMPLATE = {
    "campaign_name": "PLACEHOLDER",
    "run_scope": {"dataset_name": "PLACEHOLDER", "wells": ["PLACEHOLDER"]},
    "hpo_params": {
        "trials_per_cycle_schedule": [5, 5, 5], # This will be overridden
        "metric_to_optimize": "weighted_score",
        "metric_weights": {"val_smape_cum": 2.0, "val_smape_agg": 2.0},
        "lower_is_better": {"val_smape_cum": True, "val_smape_agg": True},
        "search_space_func_name": "define_search_space"
    },
    "job_defaults": {
        "seed": 42,
        "architecture_name": "PLACEHOLDER",
        "feature_kind": "Normal", "use_known_good": False, "lag_window": 300,"horizon": 300,
        "patience": 20, "test_size": 0.6, "val_size": 0.1,
        "aggregation_method": "median", "evaluate_by_slice": True,
        "aggregation_quantiles": [0.25, 0.5, 0.75], "plot": False,
        "scenario": "P50", "band": None, "show_components": False,
    },
    "run_params": {"ensemble_size": 1, "max_workers": 1},
    "infra": {
        "profiles_dir": "../../src/experiment_configs/profiles",
        "experiments_output_dir": "../../src/experiment_configs/results",
        "hpo_studies_dir": "../../src/experiment_configs/studies",
        "architecture_yaml_path": "../../src/experiment_configs/architectures_PINNs.yaml",
        "log_level": "INFO"
    }
}

# ==============================================================================
# 2. EXPERIMENTAL DESIGN CONTROL PANEL (The New, Powerful Part)
# ==============================================================================
# Here, you define the high-level design of your entire experimental campaign.

# --- a) Select which datasets to include in this batch ---
# The names must match the 'name' field in your `get_data_sources` function.
DATASETS_TO_RUN = ["VOLVE", "UNISIM_IV"]
DATASETS_TO_RUN = ["UNISIM_IV"]

# --- b) Define which architectures to test on the selected datasets ---
ARCHITECTURES_TO_TEST = ["Seq2Context", "Seq2Trend", "Seq2PIN"]
ARCHITECTURES_TO_TEST = ["Seq2Trend"]

# --- c) Define the HPO trial budget per architecture ---
# This allows you to give more complex models a larger search budget.
TRIALS_PER_ARCHITECTURE = {
    "Seq2Context": [100, 60, 40],
    "Seq2Trend":   [100, 60, 40],
    "Seq2PIN":     [100, 60, 40],
    "default":     [10, 5] # Fallback for any archs not listed
}


TRIALS_PER_ARCHITECTURE = {
    "Seq2Context": [60, 30, 20],
    "Seq2Trend":   [150, 100, 50],
    "Seq2PIN":     [40, 40, 40],
    "default":     [10, 5] # Fallback for any archs not listed
}

# ==============================================================================
# 3. DYNAMIC EXPERIMENT MATRIX GENERATOR
# ==============================================================================
def create_experiment_matrix() -> list:
    """
    Dynamically builds the experiment matrix based on the control panel settings.
    """
    matrix = []
    all_data_sources = get_data_sources()
    
    # Filter to only the datasets we want to run
    selected_sources = [ds for ds in all_data_sources if ds["name"] in DATASETS_TO_RUN]
    
    for source in selected_sources:
        dataset_name = source["name"]
        # Handle the comma-separated string of wells in UNISIM_IV
        wells = source.get("wells", [])
        if isinstance(wells, list) and len(wells) == 1 and "," in wells[0]:
             wells = [w.strip() for w in wells[0].split(',')]
        
        for well_name in wells:
            for arch_name in ARCHITECTURES_TO_TEST:
                matrix.append({
                    "dataset": dataset_name,
                    "well": well_name,
                    "arch": arch_name
                })
    return matrix

# ==============================================================================
# 4. MAIN GENERATOR LOGIC (Final Version)
# ==============================================================================
def generate_campaign_files():
    """Generates all campaign YAML files based on the dynamically created matrix."""
    output_dir = Path().parent
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Generating campaign files in: {output_dir.resolve()}")
    
    experiment_matrix = create_experiment_matrix()
    print(f"Dynamically generated {len(experiment_matrix)} experiments to create.")
    
    for experiment in experiment_matrix:
        config = copy.deepcopy(CAMPAIGN_TEMPLATE)
        
        ds_name, well_name, arch_name = experiment["dataset"], experiment["well"], experiment["arch"]
        
        # Populate the template
        campaign_base_name = f"{ds_name}_{well_name.replace('/', '-')}_{arch_name}"
        config["campaign_name"] = campaign_base_name
        config["run_scope"]["dataset_name"] = ds_name
        config["run_scope"]["wells"] = [well_name]
        config["job_defaults"]["architecture_name"] = arch_name
        
        # Set the trial schedule based on the architecture
        config["hpo_params"]["trials_per_cycle_schedule"] = TRIALS_PER_ARCHITECTURE.get(arch_name, TRIALS_PER_ARCHITECTURE["default"])
        
        # Architecture-aware logic for the YAML path
        if arch_name != "Seq2Context":
            del config["infra"]["architecture_yaml_path"]

        # Save the file
        output_path = output_dir / f"{campaign_base_name.lower()}.yaml"
        with open(output_path, 'w') as f:
            yaml.dump(config, f, sort_keys=False, default_flow_style=None, indent=2)
            
        print(f"  -> Created: {output_path.name}")
        
    print(f"\n✅ Successfully generated {len(experiment_matrix)} campaign files.")

if __name__ == "__main__":
    generate_campaign_files()

Generating campaign files in: /home/gabriel/Documentos/Equinor/src/experiment_configs/hpo_campaigns
Dynamically generated 6 experiments to create.
  -> Created: unisim_iv_p11_seq2trend.yaml
  -> Created: unisim_iv_p12_seq2trend.yaml
  -> Created: unisim_iv_p13_seq2trend.yaml
  -> Created: unisim_iv_p14_seq2trend.yaml
  -> Created: unisim_iv_p15_seq2trend.yaml
  -> Created: unisim_iv_p16_seq2trend.yaml

✅ Successfully generated 6 campaign files.
